# 05 — Leave-one-cohort-out split audit

**Scope:** Validate and save the four outer and twelve nested cohort-held-out partitions. No model fitting or data-dependent preprocessing.

In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

candidates = [Path(os.environ["CMPS355_ROOT"])] if os.environ.get("CMPS355_ROOT") else []
candidates.extend([
    Path("/content/drive/MyDrive/CMPS355_Research"),
    Path.cwd(),
    *Path.cwd().parents,
])
ROOT = next((p.resolve() for p in candidates if (p / ".cmps355_root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("CMPS355_Research project root not found")
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT.name)

Mounted at /content/drive
Project: CMPS355_Research


## Dataset and sample index

In [2]:
import json
import pandas as pd
from cmps355.split_design import read_sample_index, build_assignments, audit_assignments, save_assignments

manifest = json.loads((ROOT / "configs" / "dataset.json").read_text(encoding="utf-8"))
archive = ROOT / manifest["artifact"]
index = read_sample_index(archive, manifest)
print("Samples:", len(index))
print("BioProjects:", index["Study"].nunique())
display(pd.crosstab(index["cohort"], index["class"]))

Samples: 689
BioProjects: 6


class,CD,Not IBD
cohort,,
d1,218,42
d2,134,45
d3,34,42
d4,109,65


## Outer and inner holdouts

In [3]:
outer, inner = build_assignments(index, manifest["cohorts"])
report = audit_assignments(index, outer, inner, manifest["cohorts"])
assert report["status"] == "PASS"
summary = pd.DataFrame([{
    "outer_holdout": fold["test_cohort"],
    "training_samples": fold["train_samples"],
    "test_samples": fold["test_samples"],
    "inner_folds": len(fold["inner_folds"]),
    "train_CD": fold["train_class_counts"]["CD"],
    "test_CD": fold["test_class_counts"]["CD"],
} for fold in report["folds"]])
display(summary)
print("Outer folds:", report["outer_fold_count"])
print("Inner folds:", report["inner_fold_count"])

,outer_holdout,training_samples,test_samples,inner_folds,train_CD,test_CD
0,d1,429,260,3,277,218
1,d2,510,179,3,361,134
2,d3,613,76,3,461,34
3,d4,515,174,3,386,109


Outer folds: 4
Inner folds: 12


## Source-study composition

In [4]:
study_counts = pd.DataFrame(report["source_class_composition"])
display(study_counts.pivot_table(index=["cohort", "bioproject"], columns="class", values="count", aggfunc="sum", fill_value=0))

class                CD  Not IBD
cohort bioproject               
d1     PRJNA248469  218       42
d2     PRJNA565216  112       37
       PRJNA985602   22        8
d3     PRJNA702434   34       42
d4     PRJNA728646   17        5
       PRJNA961704   92       60

## Save immutable split assignments

In [5]:
target = save_assignments(ROOT, manifest["sha256"], outer, inner, report)
print("Saved:", target.relative_to(ROOT))
print("Files:", sorted(p.name for p in target.iterdir()))
print("LOCO SPLIT AUDIT: PASS")

Saved: runs/split_design/20261107T040042030201Z
Files: ['inner.csv', 'outer.csv', 'report.json']
LOCO SPLIT AUDIT: PASS
